# 5b Explore MNIST Before Training

Inspect shapes, pixels, label counts, class examples, and the boundary between the official training and test sets.

**Budget:** 15–25 minutes; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), select a Python 3.10+ kernel, and use the dependency setup below if its packages are missing. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


## Install the notebook dependencies if needed

The next cell installs the shared CPU packages into **this notebook's selected Python kernel** using `%pip`. It includes **scikit-learn**, whose import name is `sklearn`, plus NumPy, pandas, matplotlib, Pillow, and IPython.

**Why `%pip` instead of `!pip`?** `%pip` installs into the running notebook kernel's Python environment. `!pip` looks up `pip` on the shell's `PATH`, which may point to a different Python environment and leave `sklearn` unavailable here. Use `!` for operating-system commands such as `!hostname`. The setup cell calls `run_line_magic("pip", ...)`, the Python form of `%pip`, so installation can stay inside the opt-in condition.

If the packages are missing or this is a fresh environment, set `INSTALL_PACKAGES = True` and run the cell once. Only do this when installation is permitted and pip is configured to use your approved package source. If the environment is already prepared, leave it `False`. The cell finds `requirements.txt` whether the kernel starts in the repository root or `05_Jupyter`.

After a successful installation, change the flag back to `False`, use **Kernel → Restart Kernel**, and run the notebook from the top. A restart clears old imports and variables. If installation fails or the source is blocked, stop here and ask the administrator to prepare the environment or approved mirror; do not bypass TLS checks.

PyTorch/CUDA, interactive widgets, ClearML, and live gateway extras remain optional. See the [support guide](README.md#prepare-one-approved-python-environment) before enabling them. Package installation can use the network; the lesson's default dataset path remains offline.


In [ ]:
from pathlib import Path
import sys

# Change to True only when this kernel needs the section's packages installed.
INSTALL_PACKAGES = False

SETUP_HERE = Path.cwd().resolve()
SETUP_LAB = SETUP_HERE if (SETUP_HERE / "image_lab.py").is_file() else SETUP_HERE / "05_Jupyter"
REQUIREMENTS = SETUP_LAB / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError("Start this kernel in the repository root or 05_Jupyter folder with requirements.txt present.")
print("Selected kernel Python:", sys.executable)

if INSTALL_PACKAGES:
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this setup cell in the notebook's IPython Python kernel.")
    # Equivalent to %pip install -r "{REQUIREMENTS}"; targets the selected kernel.
    shell.run_line_magic("pip", f'install -r "{REQUIREMENTS}"')
    print("Check pip output for errors. After success, set INSTALL_PACKAGES = False, restart the kernel, and run from the top.")
else:
    print("Installation skipped. If imports fail, set INSTALL_PACKAGES = True and rerun this cell first.")


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


## Choose a dataset deliberately

Leave `ALLOW_DOWNLOAD = False` for a network-free first run. For actual MNIST, review the source/license notes, set it to `True`, and rerun. A cached download is reused. The official split is retained before bounded sampling; the selected dataset name, source, and shape below are the facts to put in your report. The test set is for final evaluation, not feature or model selection.


In [ ]:
ALLOW_DOWNLOAD = False
data = load_image_data("mnist", allow_download=ALLOW_DOWNLOAD,
                       cache_dir=OUTPUT / "data", limit_train=3000, limit_test=600, seed=SEED)
print("Actual dataset:", data.name)
print("Source:", data.source)
print("Provenance:", data.provenance)
print("Train:", data.train_images.shape, "Test:", data.test_images.shape)
print("Pixel range:", float(data.train_images.min()), float(data.train_images.max()))
assert len(data.train_images) == len(data.train_labels)
assert data.train_images.ndim == 3
assert 0 <= data.train_images.min() <= data.train_images.max() <= 1


## Inspect training examples and balance

Use only the training split for exploratory analysis. Row and column order, background polarity, and value ranges are part of the model's input contract. A flattened vector still has a fixed pixel order.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for label, ax in enumerate(axes.flat):
    positions = np.flatnonzero(data.train_labels == label)
    if len(positions):
        ax.imshow(data.train_images[positions[0]], cmap="gray", vmin=0, vmax=1)
    ax.set_title(data.class_names[label]); ax.axis("off")
fig.suptitle(data.name); plt.tight_layout(); plt.show()
counts = np.bincount(data.train_labels, minlength=len(data.class_names))
plt.bar(data.class_names, counts); plt.title("Training labels only")
plt.xlabel("Class"); plt.ylabel("Examples"); plt.show()
print("Flattened features:", int(np.prod(data.train_images.shape[1:])))


## Why the sample size matters

The small sample is chosen for teaching speed, not a benchmark claim. MNIST has 60,000 official training and 10,000 official test images, each 28×28. The bundled digits fallback contains fewer 8×8 images and uses its own deterministic split. Neither source should be described as a full NIST handwriting corpus. Never combine the two sources while retaining one dataset label.


## Checkpoint, exercise, and cleanup

**Checkpoint:** You can identify the actual source, pixel range, dimensions, train/test boundary, and one example per available class.

**Try it:** Compare class counts and pixel shape after switching from the fallback to real MNIST. Describe what changed before comparing any accuracy.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5c: Train a Digit Classifier](05c_Train_a_Digit_Classifier.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
